# TP Integrador: Álgebra en mis propios datos

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

df = pd.read_excel("../data/futbolargentino.xlsx", sheet_name="Worksheet")

df["Altura_num"] = pd.to_numeric(df["Altura"], errors="coerce")
df["Valor_mercado_num"] = pd.to_numeric(df["Valor de mercado"], errors="coerce")

df_algebra = df.dropna(subset=['Altura_num', 'Edad', 'Valor_mercado_num']).copy()

### Imputación de Valores Nulos mediante Vectores (Álgebra Matricial)

**Qué:** 
Aplicamos el cálculo de la **Distancia Euclidiana** (la norma del vector resta: $\vert{}\vert{}\vec{v} - \vec{u}\vert{}\vert{}$) utilizando **Álgebra Matricial** (operaciones de *broadcasting* con arreglos multidimensionales en NumPy). Esto nos permite calcular la distancia geométrica entre múltiples vectores en simultáneo sin usar bucles iterativos.

**Dónde:** 
Sobre las columnas numéricas `Edad_num` y `Altura_num` (previamente normalizadas entre 0 y 1 para que ambas variables tengan el mismo peso). El objetivo es estimar y completar los registros vacíos (`NaN` o guiones `"-"`) de la columna `Valor de mercado`.

**Para qué:** 
Nos permite resolver el problema de los datos faltantes de una manera analítica. En lugar de rellenar los precios nulos con un cero o un promedio general (lo cual sesgaría el análisis), tratamos a cada jugador como un vector bidimensional $(edad, altura)$. Mediante el cálculo matricial, el algoritmo busca instantáneamente cuál es el jugador *que sí tiene precio* más cercano geométricamente a nuestra incógnita (su "clon físico"), y le asigna ese mismo valor de mercado. Además, vectorizar el cálculo nos demostró el poder de las matrices: redujo el tiempo de procesamiento de millones de comparaciones a una fracción de segundo.

In [2]:
df.replace(r'^\s*-\s*$', np.nan, regex=True, inplace=True)
df.replace(r'^\s*$', np.nan, regex=True, inplace=True) # También limpia los espacios vacíos

#Comprobación rápida para quedarnos tranquilos:
print("Cantidad de guiones que quedaron en 'Valor de mercado':", (df['Valor de mercado'] == '-').sum())

#Conversión a numérico 
df["Altura_num"] = pd.to_numeric(df["Altura"], errors="coerce")
df["Edad_num"] = pd.to_numeric(df["Edad"], errors="coerce")
df["Valor_mercado_num"] = pd.to_numeric(df["Valor de mercado"], errors="coerce")

#Filtramos jugadores válidos (que sí tengan edad y altura para armar el vector geométrico)
df_validos = df.dropna(subset=['Edad_num', 'Altura_num']).copy()

#Normalización entre 0 y 1
for col in ['Edad_num', 'Altura_num']:
    min_v = df_validos[col].min()
    max_v = df_validos[col].max()
    df_validos[col + '_norm'] = (df_validos[col] - min_v) / (max_v - min_v)

#Dividimos los grupos
df_con_precio = df_validos.dropna(subset=['Valor_mercado_num']).copy()
df_sin_precio = df_validos[df_validos['Valor_mercado_num'].isna()].copy()

# --- OPTIMIZACIÓN MATRICIAL CON NUMPY ---

matriz_incognitos = df_sin_precio[['Edad_num_norm', 'Altura_num_norm']].values
matriz_referencias = df_con_precio[['Edad_num_norm', 'Altura_num_norm']].values

diferencia_matrices = matriz_incognitos[:, np.newaxis, :] - matriz_referencias
distancias = np.linalg.norm(diferencia_matrices, axis=2)

indices_minimos = np.argmin(distancias, axis=1)

df_sin_precio['Valor_Estimado_Algebra'] = df_con_precio.iloc[indices_minimos]['Valor_mercado_num'].values
df_sin_precio['Clon_Utilizado'] = df_con_precio.iloc[indices_minimos]['Jugadores'].values

print("¡Cálculo matricial finalizado!")
display(df_sin_precio[['Jugadores', 'Posicion', 'Edad', 'Altura', 'Clon_Utilizado', 'Valor_Estimado_Algebra']].head(10))

Cantidad de guiones que quedaron en 'Valor de mercado': 0
¡Cálculo matricial finalizado!


,Jugadores,Posicion,Edad,Altura,Clon_Utilizado,Valor_Estimado_Algebra
1,Maximiliano Sola,Defensa,22,1.8,Álvaro Pereira,2900000.0
17,Sebastián Brunet,Lateral izquierdo,24,1.74,Emanuel Coronel,1200000.0
27,Franco Quiroga,Mediocentro,21,1.77,Mariano Almandoz,75000.0
29,Andrés Romero,Interior derecho,18,1.78,Pablo Ledesma,350000.0
37,Marcos Ovejero,Extremo derecho,21,1.7,Facundo Alfonso,50000.0
45,Emmanuel Gigliotti,Delantero centro,21,1.87,Nicolás Freire,750000.0
46,Óscar Alegre,Delantero centro,22,1.76,Guillermo Pereyra,175000.0
52,Diego Quirós,Portero,20,1.79,Nicolás Batista,25000.0
59,Nicolás Berardo,Defensa central,18,1.81,Facundo Kruspzky,50000.0
109,Lucas Rodríguez,Lateral izquierdo,16,1.68,Lautaro Acosta,1200000.0


### Similitud de Perfiles Deportivos (Distancia Euclidiana)

**Qué:** 
Utilizamos el cálculo de la **Distancia Euclidiana**, que equivale a calcular la norma de la resta entre dos vectores ($\vert{}\vert{} \vec{v} - \vec{u} \vert{}\vert{}$) utilizando la función `np.linalg.norm()`. Este concepto de la Unidad 1 nos permite medir matemáticamente la distancia geométrica entre dos puntos en un espacio tridimensional.

**Dónde:** 
Sobre las columnas numéricas `Altura_num`, `Edad_num` y `Valor_mercado_num`. Primero, filtramos las filas que contenían valores nulos en estas variables. Luego, normalizamos los valores (escalándolos entre 0 y 1) para evitar que variables con magnitudes muy grandes (como el valor de mercado en millones) distorsionen el cálculo frente a variables de magnitud pequeña (como la altura en metros).

**Para qué:** 
Este enfoque convierte a cada jugador en un vector en $\mathbb{R}^3$, lo que nos permite desarrollar un algoritmo básico de *scouting* (búsqueda de talentos). Al tomar un jugador de referencia, el cálculo de la distancia euclidiana frente al resto del dataset identifica qué futbolistas tienen el perfil estadístico más parecido (menor distancia). Para evitar resultados redundantes, excluimos los registros históricos del mismo jugador, obteniendo así a sus verdaderos "clones" matemáticos en términos de edad, altura y precio.

In [4]:
df["Altura_num"] = pd.to_numeric(df["Altura"], errors="coerce")
df["Valor_mercado_num"] = pd.to_numeric(df["Valor de mercado"], errors="coerce")
df["Edad_num"] = pd.to_numeric(df["Edad"], errors="coerce")

#Filtramos los valores nulos SOLO para el dataset de álgebra
df_algebra = df.dropna(subset=['Altura_num', 'Edad_num', 'Valor_mercado_num']).copy()

#NORMALIZACIÓN (para que el Valor de mercado no "pise" a la Altura al calcular)
for col in ['Altura_num', 'Edad_num', 'Valor_mercado_num']:
    min_val = df_algebra[col].min()
    max_val = df_algebra[col].max()
    df_algebra[f'{col}_norm'] = (df_algebra[col] - min_val) / (max_val - min_val)

#ALGEBRA UNIDAD 1: Distancia Euclidiana (Norma del vector)
indice_referencia = 10

#guardamos los datos del jugador de referencia
jugador_nombre = df_algebra.iloc[indice_referencia]['Jugadores']
jugador_posicion = df_algebra.iloc[indice_referencia]['Posicion']

#Vector del jugador de referencia
jugador_referencia = df_algebra.iloc[indice_referencia][['Altura_num_norm', 'Edad_num_norm', 'Valor_mercado_num_norm']].values

distancias = []
for index, row in df_algebra.iterrows():
    vector_jugador = row[['Altura_num_norm', 'Edad_num_norm', 'Valor_mercado_num_norm']].values
    distancia = np.linalg.norm(jugador_referencia - vector_jugador)
    distancias.append(distancia)

df_algebra['Distancia_al_jugador'] = distancias

#FILTRO: Excluimos al propio jugador de los resultados (para evitar sus registros de otros años)
df_sin_clones_exactos = df_algebra[df_algebra['Jugadores'] != jugador_nombre]

#Ordenamos y mostramos los 8 más parecidos
jugadores_similares = df_sin_clones_exactos.sort_values('Distancia_al_jugador').head(8)

print(f"Buscando similitudes matemáticas para: {jugador_nombre} ({jugador_posicion})")
print("Excluyendo registros históricos del mismo jugador.")
print("-" * 75)

display(jugadores_similares[['Jugadores', 'Posicion', 'Edad', 'Altura_num', 'Valor de mercado', 'Distancia_al_jugador']])

Buscando similitudes matemáticas para: Ignacio Canuto (Defensa central)
Excluyendo registros históricos del mismo jugador.
---------------------------------------------------------------------------


,Jugadores,Posicion,Edad,Altura_num,Valor de mercado,Distancia_al_jugador
7967,Nicolás Castro,Mediocentro ofensivo,22,1.82,100000,0.000000
4246,Nacho Fernández,Mediocentro ofensivo,22,1.82,75000,0.001000
4413,Rodrigo Contreras,Delantero centro,22,1.82,75000,0.001000
10457,Rodrigo Contreras,Delantero centro,22,1.82,75000,0.001000
5159,Ramiro Martínez,Portero,22,1.82,75000,0.001000
8829,Gonzalo Castillejos,Delantero centro,22,1.82,50000,0.002001
5602,Mauro Villegas,Delantero centro,22,1.82,150000,0.002001
5882,Fernando Cosciuc,Defensa central,22,1.82,50000,0.002001


# Índice de Fichaje Personalizado (Producto Escalar)

**Qué:** Aplicamos el concepto de Producto Escalar (o Dot Product) de la Unidad 1. Es la multiplicación componente a componente de dos vectores y la suma de esos resultados ($\vec{u} \cdot \vec{v}$). El código utiliza la función np.dot() de NumPy.

**Dónde:** Se calcula multiplicando el vector numérico normalizado de cada jugador (Altura_num_norm, Edad_num_norm, Valor_mercado_num_norm) por un vector arbitrario de "pesos" que definimos nosotros, el cual representa las prioridades del club (por ejemplo, premiar la altura con un número positivo y penalizar la edad y el precio con números negativos).

**Para qué:** Permite transformar las estadísticas sueltas en un "Score" o puntaje único y personalizado para ordenar el dataset. A diferencia de un simple filtro de Excel, el producto escalar pondera todas las variables al mismo tiempo según su importancia matemática, permitiendo descubrir jugadores infravalorados que cumplen exactamente con los requerimientos específicos que busca la dirigencia de un club.

In [5]:
#Definimos el vector de requerimientos(pesos)
#Buscamos mucha altura (peso 0.6), castigamos la edad alta (-0.4) y buscamos que sean baratos (-0.5)
vector_pesos = np.array([0.6, -0.4, -0.5])

puntajes = []
for index, row in df_algebra.iterrows():
    #Vector del jugador en el espacio R3
    vector_jugador = row[['Altura_num_norm', 'Edad_num_norm', 'Valor_mercado_num_norm']].values
    
    #Producto escalar entre el perfil del jugador y nuestros requerimientos
    score = np.dot(vector_jugador, vector_pesos)
    puntajes.append(score)

df_algebra['Puntaje_Scouting'] = puntajes

#Mostramos los mejores candidatos
mejores_fichajes = df_algebra.sort_values('Puntaje_Scouting', ascending=False).head(5)

print("Top 5 candidatos según el Producto Escalar (Índice de Fichaje):")
display(mejores_fichajes[['Jugadores', 'Posicion', 'Edad', 'Altura_num', 'Valor de mercado', 'Puntaje_Scouting']])

Top 5 candidatos según el Producto Escalar (Índice de Fichaje):


,Jugadores,Posicion,Edad,Altura_num,Valor de mercado,Puntaje_Scouting
5924,Fernando Pellegrino,Portero,20,1.98,200000,0.506569
3006,Santiago Fosgt,Defensa central,20,1.97,650000,0.483612
7985,Williams Barlasina,Portero,19,1.95,25000,0.479021
7143,Esteban Andrada,Portero,17,1.94,450000,0.478186
6830,Esteban Andrada,Portero,18,1.94,25000,0.475878


# Medición de Diversidad del Plantel (Determinantes)

**Qué:** Utilizamos el cálculo del Determinante de una matriz de 3x3 ($\det(A)$) correspondiente a la Unidad 3. Desde su interpretación geométrica, el valor absoluto del determinante representa el volumen del paralelepípedo formado por tres vectores en el espacio. El código utiliza np.linalg.det().

**Dónde:** Se aplica tomando los datos numéricos de 3 jugadores que jueguen en la misma posición (por ejemplo, 3 defensores). Convertimos a cada jugador en una fila, formando una matriz cuadrada de $3 \times 3$.

**Para qué:** Nos sirve para evaluar matemáticamente si un club tiene perfiles repetidos. Si el determinante es cero (o muy cercano a cero), los vectores son linealmente dependientes (coplanares); esto indica que los tres jugadores tienen perfiles estadísticos idénticos (el plantel es redundante). Si el determinante es alto, los vectores son linealmente independientes, lo que significa que el técnico cuenta con opciones muy variadas y diversas para ese mismo puesto.

In [6]:
# 1. Definimos los filtros (Podés cambiar estos textos por el equipo y año que quieras analizar)
club_elegido = "Boca" 
temporada_elegida = 2020

# 2. Filtramos el dataset por Club, Temporada y que sean Delanteros Centro
delanteros_equipo = df_algebra[
    (df_algebra['Club'] == club_elegido) & 
    (df_algebra['Temporada'] == temporada_elegida) & 
    (df_algebra['Posicion'].str.contains('Delantero centro', na=False, case=False))
]

# 3. Verificamos si quedaron al menos 3 jugadores para poder armar la matriz 3x3
if len(delanteros_equipo) >= 3:
    
    # Tomamos los primeros 3 delanteros de ese equipo específico
    jugador_1 = delanteros_equipo.iloc[0][['Altura_num_norm', 'Edad_num_norm', 'Valor_mercado_num_norm']].values
    jugador_2 = delanteros_equipo.iloc[1][['Altura_num_norm', 'Edad_num_norm', 'Valor_mercado_num_norm']].values
    jugador_3 = delanteros_equipo.iloc[2][['Altura_num_norm', 'Edad_num_norm', 'Valor_mercado_num_norm']].values

    # Armamos la matriz 3x3 forzando el formato numérico
    matriz_tridente = np.array([jugador_1, jugador_2, jugador_3], dtype=float)

    # ÁLGEBRA UNIDAD 3: Cálculo del determinante
    determinante = np.linalg.det(matriz_tridente)
    volumen = abs(determinante)

    # Mostramos los resultados
    nombres = f"{delanteros_equipo.iloc[0]['Jugadores']}, {delanteros_equipo.iloc[1]['Jugadores']} y {delanteros_equipo.iloc[2]['Jugadores']}"
    
    print(f"Analizando el ataque de: {club_elegido} (Temporada {temporada_elegida})")
    print(f"Tridente evaluado: {nombres}")
    print("-" * 75)
    print(f"El determinante de la matriz es: {determinante:.4f}")
    print(f"El 'Volumen de Diversidad' es: {volumen:.4f}\n")

    if volumen < 0.05:
        print("Conclusión Matemática: Volumen cercano a 0. Vectores casi linealmente dependientes.")
        print("Conclusión Deportiva: Los 3 delanteros son perfiles idénticos. Hay redundancia en el puesto.")
    else:
        print("Conclusión Matemática: Volumen considerable. Vectores linealmente independientes.")
        print("Conclusión Deportiva: El equipo tiene delanteros con perfiles físicos y económicos muy diversos.")
        
else:
    print(f"No hay suficientes delanteros centro en {club_elegido} para la temporada {temporada_elegida} con los datos completos para armar una matriz 3x3.")

Analizando el ataque de: Boca (Temporada 2020)
Tridente evaluado: Jan Hurtado, Ramón Ábila y Mauro Zárate
---------------------------------------------------------------------------
El determinante de la matriz es: -0.0121
El 'Volumen de Diversidad' es: 0.0121

Conclusión Matemática: Volumen cercano a 0. Vectores casi linealmente dependientes.
Conclusión Deportiva: Los 3 delanteros son perfiles idénticos. Hay redundancia en el puesto.


# Conclusion

Lo que más nos sirvió en este trabajo fue combinar vectores y matrices (Unidades 1 y 2). Pensar a los jugadores como vectores nos permitió armar un buscador de talentos real, y usar operaciones matriciales nos salvó para calcular los datos nulos en un segundo sin que se cuelgue la compu. Por el contrario, los determinantes (Unidad 3) fueron lo menos práctico: la idea de medir la diversidad de un plantel está buena, pero te obliga a tener exactamente 3 jugadores perfectos del mismo puesto y club, lo que lo hace casi imposible de automatizar a gran escala.